# Train a model of human choices

Fine-tunes Qwen2.5-0.5B on a sample of Psych-101 (real choices from 160 psychology experiments) so
it predicts what people choose, then tests it on Oxpecker's workplace decisions.

Choose **Runtime → Change runtime type → T4 GPU** first, then run the cells in order.

In [ ]:
BASE = "Qwen/Qwen2.5-0.5B"
RUN = "runs/human"
HF_REPO = "EthanChen72/human-choice-qwen2.5-0.5b-lora"  # where the trained model is published
UPLOAD_OXPECKER = False  # set to True to upload Oxpecker research transcripts in step 2


In [ ]:
!git clone --depth 1 https://github.com/vittesh12345/Oxpecker
%cd Oxpecker
!pip install -q -r training/requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

Download Psych-101 (about 860 MB) and keep a sample from every experiment. Takes a few minutes.

In [ ]:
!python training/human/prepare.py --output {RUN}/data

Optional: upload the Oxpecker transcripts file (**Practice log → Recorded answers → Download
transcripts** in the app) to test the model on real workplace decisions. Set `UPLOAD_OXPECKER = True` in the
first cell to be asked for the file; otherwise this step is skipped.

In [ ]:
if UPLOAD_OXPECKER:
    from google.colab import files
    files.upload()


In [ ]:
import os
OXPECKER = "--oxpecker oxpecker-transcripts.jsonl" if os.path.exists("oxpecker-transcripts.jsonl") else ""
print("Testing on Oxpecker decisions too" if OXPECKER else "No Oxpecker file: testing on Psych-101 only")

Score the base model first, so there is a before-and-after comparison.

In [ ]:
!python training/human/evaluate.py --base-model {BASE} --psych101 {RUN}/data/heldout.jsonl {OXPECKER} --output {RUN}/eval-base.json

Fine-tune. This is the long step.

In [ ]:
!python training/human/train.py --base-model {BASE} --data {RUN}/data/train.jsonl --output {RUN}

In [ ]:
!python training/human/evaluate.py --base-model {BASE} --adapter {RUN}/adapter --psych101 {RUN}/data/heldout.jsonl {OXPECKER} --output {RUN}/eval-tuned.json

Log in with a Hugging Face token that has write access, then publish the adapter with its scores.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python training/human/publish.py --run {RUN} --repo {HF_REPO}